In [2]:
!pip install -q -U ultralytics

import ultralytics
print(ultralytics.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.6 MB/s eta 0:00:00a 0:00:01
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
8.4.33


In [3]:
# 定义 ECA 模块，并 patch parse_model()
import torch
import torch.nn as nn
import contextlib
import ast

import ultralytics.nn.tasks as tasks
from ultralytics.nn.tasks import (
    Classify, Conv, ConvTranspose, GhostConv, Bottleneck, GhostBottleneck,
    SPP, SPPF, C2fPSA, C2PSA, DWConv, Focus, BottleneckCSP, C1, C2, C2f,
    C3k2, RepNCSPELAN4, ELAN1, ADown, AConv, SPPELAN, C2fAttn, C3, C3TR,
    C3Ghost, DWConvTranspose2d, C3x, RepC3, PSA, SCDown, C2fCIB, A2C2f,
    AIFI, HGStem, HGBlock, ResNetLayer, Concat, Detect, WorldDetect,
    YOLOEDetect, Segment, Segment26, YOLOESegment, YOLOESegment26,
    Pose, Pose26, OBB, OBB26, v10Detect, ImagePoolingAttn,
    RTDETRDecoder, CBLinear, CBFuse
)
from ultralytics.utils import LOGGER, colorstr
from ultralytics.utils.ops import make_divisible


class ECA(nn.Module):
    """
    Efficient Channel Attention
    input : [B, C, H, W]
    output: [B, C, H, W]
    """
    def __init__(self, c1, k_size=3):
        super().__init__()
        assert k_size % 2 == 1, "ECA kernel size should be odd, e.g. 3/5/7"
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(
            in_channels=1,
            out_channels=1,
            kernel_size=k_size,
            padding=(k_size - 1) // 2,
            bias=False
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x: [B, C, H, W]
        y = self.avg_pool(x)                       # [B, C, 1, 1]
        y = y.squeeze(-1).transpose(-1, -2)       # [B, 1, C]
        y = self.conv(y)                          # [B, 1, C]
        y = y.transpose(-1, -2).unsqueeze(-1)     # [B, C, 1, 1]
        y = self.sigmoid(y)
        return x * y.expand_as(x)


# 让 YAML 里的 "ECA" 能被 tasks.__dict__[m] 找到
tasks.ECA = ECA


def parse_model_eca(d, ch, verbose=True):
    """Patch version of parse_model with ECA support."""
    legacy = True
    max_channels = float("inf")
    nc, act, scales, end2end = (d.get(x) for x in ("nc", "activation", "scales", "end2end"))
    reg_max = d.get("reg_max", 16)
    depth, width, kpt_shape = (d.get(x, 1.0) for x in ("depth_multiple", "width_multiple", "kpt_shape"))
    scale = d.get("scale")

    if scales:
        if not scale:
            scale = next(iter(scales.keys()))
            LOGGER.warning(f"no model scale passed. Assuming scale='{scale}'.")
        depth, width, max_channels = scales[scale]

    if act:
        Conv.default_act = eval(act)
        if verbose:
            LOGGER.info(f"{colorstr('activation:')} {act}")

    if verbose:
        LOGGER.info(f"\n{'':>3}{'from':>20}{'n':>3}{'params':>10}  {'module':<45}{'arguments':<30}")

    ch = [ch]
    layers, save, c2 = [], [], ch[-1]

    base_modules = frozenset({
        Classify, Conv, ConvTranspose, GhostConv, Bottleneck, GhostBottleneck,
        SPP, SPPF, C2fPSA, C2PSA, DWConv, Focus, BottleneckCSP, C1, C2, C2f,
        C3k2, RepNCSPELAN4, ELAN1, ADown, AConv, SPPELAN, C2fAttn, C3, C3TR,
        C3Ghost, torch.nn.ConvTranspose2d, DWConvTranspose2d, C3x, RepC3, PSA,
        SCDown, C2fCIB, A2C2f
    })

    repeat_modules = frozenset({
        BottleneckCSP, C1, C2, C2f, C3k2, C2fAttn, C3, C3TR, C3Ghost,
        C3x, RepC3, C2fPSA, C2fCIB, C2PSA, A2C2f
    })

    for i, (f, n, m, args) in enumerate(d["backbone"] + d["head"]):
        m = (
            getattr(torch.nn, m[3:]) if "nn." in m
            else getattr(__import__("torchvision").ops, m[16:]) if "torchvision.ops." in m
            else tasks.__dict__[m]
        )

        for j, a in enumerate(args):
            if isinstance(a, str):
                with contextlib.suppress(ValueError):
                    args[j] = locals()[a] if a in locals() else ast.literal_eval(a)

        n = n_ = max(round(n * depth), 1) if n > 1 else n

        # ===== 关键：单独处理 ECA =====
        if m is ECA:
            c1 = ch[f]
            k = args[0] if len(args) else 3   # YAML 里写 [3] / [5] / [7]
            args = [c1, k]
            c2 = c1

        elif m in base_modules:
            c1, c2 = ch[f], args[0]
            if c2 != nc:
                c2 = make_divisible(min(c2, max_channels) * width, 8)

            if m is C2fAttn:
                args[1] = make_divisible(min(args[1], max_channels // 2) * width, 8)
                args[2] = int(max(round(min(args[2], max_channels // 2 // 32)) * width, 1) if args[2] > 1 else args[2])

            args = [c1, c2, *args[1:]]
            if m in repeat_modules:
                args.insert(2, n)
                n = 1

            if m is C3k2:
                legacy = False
                if scale in "mlx":
                    args[3] = True
            if m is A2C2f:
                legacy = False
                if scale in "lx":
                    args.extend((True, 1.2))
            if m is C2fCIB:
                legacy = False

        elif m is AIFI:
            args = [ch[f], *args]

        elif m in frozenset({HGStem, HGBlock}):
            c1, cm, c2 = ch[f], args[0], args[1]
            args = [c1, cm, c2, *args[2:]]
            if m is HGBlock:
                args.insert(4, n)
                n = 1

        elif m is ResNetLayer:
            c2 = args[1] if args[3] else args[1] * 4

        elif m is torch.nn.BatchNorm2d:
            args = [ch[f]]

        elif m is Concat:
            c2 = sum(ch[x] for x in f)

        elif m in frozenset({Detect, WorldDetect, YOLOEDetect, Segment, Segment26,
                             YOLOESegment, YOLOESegment26, Pose, Pose26, OBB, OBB26}):
            args.extend([reg_max, end2end, [ch[x] for x in f]])
            if m in {Segment, YOLOESegment, Segment26, YOLOESegment26}:
                args[2] = make_divisible(min(args[2], max_channels) * width, 8)
            if m in {Detect, YOLOEDetect, Segment, Segment26, YOLOESegment, YOLOESegment26, Pose, Pose26, OBB, OBB26}:
                m.legacy = legacy

        elif m is v10Detect:
            args.append([ch[x] for x in f])

        elif m is ImagePoolingAttn:
            args.insert(1, [ch[x] for x in f])

        elif m is RTDETRDecoder:
            args.insert(1, [ch[x] for x in f])

        elif m is CBLinear:
            c2 = args[0]
            c1 = ch[f]
            args = [c1, c2, *args[1:]]

        elif m is CBFuse:
            c2 = ch[f[-1]]

        else:
            c2 = ch[f]

        m_ = torch.nn.Sequential(*(m(*args) for _ in range(n))) if n > 1 else m(*args)
        t = str(m)[8:-2].replace("__main__.", "")
        m_.np = sum(x.numel() for x in m_.parameters())
        m_.i, m_.f, m_.type = i, f, t

        if verbose:
            LOGGER.info(f"{i:>3}{str(f):>20}{n_:>3}{m_.np:10.0f}  {t:<45}{str(args):<30}")

        save.extend(x % i for x in ([f] if isinstance(f, int) else f) if x != -1)
        layers.append(m_)
        if i == 0:
            ch = []
        ch.append(c2)

    return torch.nn.Sequential(*layers), sorted(save)


# 替换官方 parse_model
tasks.parse_model = parse_model_eca

print("parse_model patched with ECA support")

parse_model patched with ECA support


In [4]:
# 在 backbone 的 P3 / P4 后各加 1 个 ECA。
eca_yaml = r"""
nc: 3

scales:
  n: [0.33, 0.25, 1024]
  s: [0.33, 0.50, 1024]
  m: [0.67, 0.75, 768]
  l: [1.00, 1.00, 512]
  x: [1.00, 1.25, 512]

backbone:
  - [-1, 1, Conv, [64, 3, 2]]        # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]]       # 1-P2/4
  - [-1, 3, C2f, [128, True]]        # 2
  - [-1, 1, Conv, [256, 3, 2]]       # 3-P3/8
  - [-1, 6, C2f, [256, True]]        # 4
  - [-1, 1, ECA, [3]]                # 5  <-- after P3

  - [-1, 1, Conv, [512, 3, 2]]       # 6-P4/16
  - [-1, 6, C2f, [512, True]]        # 7
  - [-1, 1, ECA, [3]]                # 8  <-- after P4

  - [-1, 1, Conv, [1024, 3, 2]]      # 9-P5/32
  - [-1, 3, C2f, [1024, True]]       # 10
  - [-1, 1, SPPF, [1024, 5]]         # 11

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]  # 12
  - [[-1, 8], 1, Concat, [1]]                   # 13
  - [-1, 3, C2f, [512]]                         # 14

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]  # 15
  - [[-1, 5], 1, Concat, [1]]                   # 16
  - [-1, 3, C2f, [256]]                         # 17 (small)

  - [-1, 1, Conv, [256, 3, 2]]                  # 18
  - [[-1, 14], 1, Concat, [1]]                  # 19
  - [-1, 3, C2f, [512]]                         # 20 (medium)

  - [-1, 1, Conv, [512, 3, 2]]                  # 21
  - [[-1, 11], 1, Concat, [1]]                  # 22
  - [-1, 3, C2f, [1024]]                        # 23 (large)

  - [[17, 20, 23], 1, Detect, [nc]]             # 24
"""

yaml_path = "/kaggle/working/yolov8n_eca_p34.yaml"
with open(yaml_path, "w", encoding="utf-8") as f:
    f.write(eca_yaml)

print(open(yaml_path, "r", encoding="utf-8").read())


nc: 3

scales:
  n: [0.33, 0.25, 1024]
  s: [0.33, 0.50, 1024]
  m: [0.67, 0.75, 768]
  l: [1.00, 1.00, 512]
  x: [1.00, 1.25, 512]

backbone:
  - [-1, 1, Conv, [64, 3, 2]]        # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]]       # 1-P2/4
  - [-1, 3, C2f, [128, True]]        # 2
  - [-1, 1, Conv, [256, 3, 2]]       # 3-P3/8
  - [-1, 6, C2f, [256, True]]        # 4
  - [-1, 1, ECA, [3]]                # 5  <-- after P3

  - [-1, 1, Conv, [512, 3, 2]]       # 6-P4/16
  - [-1, 6, C2f, [512, True]]        # 7
  - [-1, 1, ECA, [3]]                # 8  <-- after P4

  - [-1, 1, Conv, [1024, 3, 2]]      # 9-P5/32
  - [-1, 3, C2f, [1024, True]]       # 10
  - [-1, 1, SPPF, [1024, 5]]         # 11

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]  # 12
  - [[-1, 8], 1, Concat, [1]]                   # 13
  - [-1, 3, C2f, [512]]                         # 14

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]  # 15
  - [[-1, 5], 1, Concat, [1]]                   # 16
  - [-1, 3, C2f, [256]]         

In [5]:
import os

candidates = []
for root, dirs, files in os.walk("/kaggle/input"):
    need = [
        os.path.join(root, "train", "images"),
        os.path.join(root, "train", "labels"),
        os.path.join(root, "val", "images"),
        os.path.join(root, "val", "labels"),
    ]
    if all(os.path.exists(p) for p in need):
        candidates.append(root)

print("Candidates:")
for c in candidates:
    print(c)

Candidates:
/kaggle/input/datasets/zhixianwan/phi-yolo/phi_yolo
/kaggle/input/datasets/zhixianwan/phi-yolo/phi_set
/kaggle/input/datasets/zhixianwan/phi-yolo/phiset


In [6]:
# data.yaml
dataset_root = "/kaggle/input/datasets/zhixianwan/phi-yolo/phi_set"

data_yaml = f"""
path: {dataset_root}
train: train/images
val: val/images
test: test/images

names:
  0: patient_info
  1: time_info
  2: institution_info
"""

with open("/kaggle/working/data.yaml", "w", encoding="utf-8") as f:
    f.write(data_yaml)

print(open("/kaggle/working/data.yaml", "r", encoding="utf-8").read())


path: /kaggle/input/datasets/zhixianwan/phi-yolo/phi_set
train: train/images
val: val/images
test: test/images

names:
  0: patient_info
  1: time_info
  2: institution_info



In [7]:
# 存在性检查
import os

print("yaml exists:", os.path.exists("/kaggle/working/data.yaml"))
print("train/images:", os.path.exists(f"{dataset_root}/train/images"))
print("train/labels:", os.path.exists(f"{dataset_root}/train/labels"))
print("val/images:", os.path.exists(f"{dataset_root}/val/images"))
print("val/labels:", os.path.exists(f"{dataset_root}/val/labels"))
print("test/images:", os.path.exists(f"{dataset_root}/test/images"))
print("test/labels:", os.path.exists(f"{dataset_root}/test/labels"))

yaml exists: True
train/images: True
train/labels: True
val/images: True
val/labels: True
test/images: True
test/labels: True


In [8]:
# 检查模型能不能成功建构
from ultralytics import YOLO

model = YOLO("/kaggle/working/yolov8n_eca_p34.yaml")
model.info()

YOLOv8n_eca_p34 summary: 136 layers, 3,011,439 parameters, 3,011,423 gradients, 8.2 GFLOPs


(136, 3011439, 3011423, 8.198144000000001)

In [9]:
# 加载预训练权重并训练
from ultralytics import YOLO

model = YOLO("/kaggle/working/yolov8n_eca_p34.yaml")
model.load("yolov8n.pt")

results = model.train(
    data="/kaggle/working/data.yaml",
    epochs=80,
    imgsz=640,
    batch=8,
    device=0,
    workers=2,
    patience=25,
    project="/kaggle/working/runs",
    name="phi_yolov8n_eca_p34",

    fliplr=0.3,
    flipud=0.15,
    mosaic=0.5,
    mixup=0.0,
    copy_paste=0.0,

    degrees=0.0,
    translate=0.02,
    scale=0.10,
    shear=0.0,
    perspective=0.0,

    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,

    plots=True,
    save=True,
    val=True
)

Transferred 78/357 items from pretrained weights
Ultralytics 8.4.33 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.3, flipud=0.15, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/working/yolov8n_eca_p34.yaml, momentum=0.937, mosaic=0.5, multi_scale=0.0, name=phi_yolov8n_eca_p34, nbs=64, nms=False, opset=

In [1]:
# 在NECK部分的P3后面加入ECA
!pip install -q -U ultralytics

import ultralytics
print(ultralytics.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.7 MB/s eta 0:00:00a 0:00:01
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
8.4.33


In [2]:
import ast
import contextlib
import torch
import torch.nn as nn

import ultralytics.nn.tasks as tasks
from ultralytics.nn.modules import Conv, C2f, SPPF, Concat, Detect
from ultralytics.utils import LOGGER, colorstr
from ultralytics.utils.ops import make_divisible


class ECA(nn.Module):
    """Efficient Channel Attention"""
    def __init__(self, c1, k_size=3):
        super().__init__()
        assert k_size % 2 == 1, "k_size must be odd, e.g. 3/5/7"
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(
            in_channels=1,
            out_channels=1,
            kernel_size=k_size,
            padding=(k_size - 1) // 2,
            bias=False
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        y = self.avg_pool(x)                   # [B, C, 1, 1]
        y = y.squeeze(-1).transpose(-1, -2)   # [B, 1, C]
        y = self.conv(y)                      # [B, 1, C]
        y = y.transpose(-1, -2).unsqueeze(-1) # [B, C, 1, 1]
        y = self.sigmoid(y)
        return x * y.expand_as(x)


# 注册到 tasks 命名空间，让 YAML 里的 "ECA" 能识别
tasks.ECA = ECA


def parse_model_eca_neck_p3(d, ch, verbose=True):
    """Minimal parse_model patch for this custom YOLOv8n + ECA YAML."""
    legacy = True
    max_channels = float("inf")

    nc, act, scales, end2end = (d.get(x) for x in ("nc", "activation", "scales", "end2end"))
    reg_max = d.get("reg_max", 16)
    depth = d.get("depth_multiple", 1.0)
    width = d.get("width_multiple", 1.0)
    scale = d.get("scale")

    if scales:
        if not scale:
            scale = next(iter(scales.keys()))
            LOGGER.warning(f"no model scale passed. Assuming scale='{scale}'.")
        depth, width, max_channels = scales[scale]

    if act:
        Conv.default_act = eval(act)
        if verbose:
            LOGGER.info(f"{colorstr('activation:')} {act}")

    if verbose:
        LOGGER.info(f"\n{'':>3}{'from':>20}{'n':>3}{'params':>10}  {'module':<35}{'arguments':<30}")

    ch = [ch]
    layers, save = [], []
    base_modules = frozenset({Conv, C2f, SPPF})
    repeat_modules = frozenset({C2f})

    for i, (f, n, m, args) in enumerate(d["backbone"] + d["head"]):
        m = getattr(nn, m[3:]) if isinstance(m, str) and m.startswith("nn.") else tasks.__dict__[m]

        for j, a in enumerate(args):
            if isinstance(a, str):
                with contextlib.suppress(Exception):
                    args[j] = locals()[a] if a in locals() else ast.literal_eval(a)

        n = max(round(n * depth), 1) if n > 1 else n

        if m is ECA:
            c1 = ch[f]
            k = args[0] if len(args) else 3
            args = [c1, k]
            c2 = c1

        elif m in base_modules:
            c1, c2 = ch[f], args[0]
            if c2 != nc:
                c2 = make_divisible(min(c2, max_channels) * width, 8)
            args = [c1, c2, *args[1:]]
            if m in repeat_modules:
                args.insert(2, n)
                n = 1

        elif m is Concat:
            c2 = sum(ch[x] for x in f)

        elif m is Detect:
            args.extend([reg_max, end2end, [ch[x] for x in f]])
            m.legacy = legacy
            c2 = None

        elif m is nn.Upsample:
            c2 = ch[f]

        elif m is nn.BatchNorm2d:
            args = [ch[f]]
            c2 = ch[f]

        else:
            c2 = ch[f]

        m_ = nn.Sequential(*(m(*args) for _ in range(n))) if n > 1 else m(*args)
        t = str(m)[8:-2].replace("__main__.", "")
        m_.np = sum(x.numel() for x in m_.parameters())
        m_.i, m_.f, m_.type = i, f, t

        if verbose:
            LOGGER.info(f"{i:>3}{str(f):>20}{n:>3}{m_.np:10.0f}  {t:<35}{str(args):<30}")

        save.extend(x % i for x in ([f] if isinstance(f, int) else f) if x != -1)
        layers.append(m_)
        if i == 0:
            ch = []
        ch.append(c2)

    return nn.Sequential(*layers), sorted(save)


tasks.parse_model = parse_model_eca_neck_p3
print("parse_model patched with ECA-neck-P3 support")

parse_model patched with ECA-neck-P3 support


In [3]:
eca_neck_p3_yaml = r"""
nc: 3

scales:
  n: [0.33, 0.25, 1024]
  s: [0.33, 0.50, 1024]
  m: [0.67, 0.75, 768]
  l: [1.00, 1.00, 512]
  x: [1.00, 1.25, 512]

backbone:
  - [-1, 1, Conv, [64, 3, 2]]        # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]]       # 1-P2/4
  - [-1, 3, C2f, [128, True]]        # 2
  - [-1, 1, Conv, [256, 3, 2]]       # 3-P3/8
  - [-1, 6, C2f, [256, True]]        # 4
  - [-1, 1, Conv, [512, 3, 2]]       # 5-P4/16
  - [-1, 6, C2f, [512, True]]        # 6
  - [-1, 1, Conv, [1024, 3, 2]]      # 7-P5/32
  - [-1, 3, C2f, [1024, True]]       # 8
  - [-1, 1, SPPF, [1024, 5]]         # 9

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]   # 10
  - [[-1, 6], 1, Concat, [1]]                    # 11
  - [-1, 3, C2f, [512]]                          # 12

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]   # 13
  - [[-1, 4], 1, Concat, [1]]                    # 14
  - [-1, 3, C2f, [256]]                          # 15  small branch
  - [-1, 1, ECA, [3]]                            # 16  <-- only neck P3

  - [-1, 1, Conv, [256, 3, 2]]                   # 17
  - [[-1, 12], 1, Concat, [1]]                   # 18
  - [-1, 3, C2f, [512]]                          # 19  medium branch

  - [-1, 1, Conv, [512, 3, 2]]                   # 20
  - [[-1, 9], 1, Concat, [1]]                    # 21
  - [-1, 3, C2f, [1024]]                         # 22  large branch

  - [[16, 19, 22], 1, Detect, [nc]]              # 23
"""

yaml_path = "/kaggle/working/yolov8n_eca_neck_p3.yaml"
with open(yaml_path, "w", encoding="utf-8") as f:
    f.write(eca_neck_p3_yaml)

print(open(yaml_path, "r", encoding="utf-8").read())


nc: 3

scales:
  n: [0.33, 0.25, 1024]
  s: [0.33, 0.50, 1024]
  m: [0.67, 0.75, 768]
  l: [1.00, 1.00, 512]
  x: [1.00, 1.25, 512]

backbone:
  - [-1, 1, Conv, [64, 3, 2]]        # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]]       # 1-P2/4
  - [-1, 3, C2f, [128, True]]        # 2
  - [-1, 1, Conv, [256, 3, 2]]       # 3-P3/8
  - [-1, 6, C2f, [256, True]]        # 4
  - [-1, 1, Conv, [512, 3, 2]]       # 5-P4/16
  - [-1, 6, C2f, [512, True]]        # 6
  - [-1, 1, Conv, [1024, 3, 2]]      # 7-P5/32
  - [-1, 3, C2f, [1024, True]]       # 8
  - [-1, 1, SPPF, [1024, 5]]         # 9

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]   # 10
  - [[-1, 6], 1, Concat, [1]]                    # 11
  - [-1, 3, C2f, [512]]                          # 12

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]   # 13
  - [[-1, 4], 1, Concat, [1]]                    # 14
  - [-1, 3, C2f, [256]]                          # 15  small branch
  - [-1, 1, ECA, [3]]                            # 16  <-- only neck P3



In [4]:
dataset_root = "/kaggle/input/datasets/zhixianwan/phi-yolo/phi_set"

data_yaml = f"""
path: {dataset_root}
train: train/images
val: val/images
test: test/images

names:
  0: patient_info
  1: time_info
  2: institution_info
"""

with open("/kaggle/working/data.yaml", "w", encoding="utf-8") as f:
    f.write(data_yaml)

print(open("/kaggle/working/data.yaml", "r", encoding="utf-8").read())


path: /kaggle/input/datasets/zhixianwan/phi-yolo/phi_set
train: train/images
val: val/images
test: test/images

names:
  0: patient_info
  1: time_info
  2: institution_info



In [5]:
import os

print("data.yaml:", os.path.exists("/kaggle/working/data.yaml"))
print("train/images:", os.path.exists(f"{dataset_root}/train/images"))
print("train/labels:", os.path.exists(f"{dataset_root}/train/labels"))
print("val/images:", os.path.exists(f"{dataset_root}/val/images"))
print("val/labels:", os.path.exists(f"{dataset_root}/val/labels"))
print("test/images:", os.path.exists(f"{dataset_root}/test/images"))
print("test/labels:", os.path.exists(f"{dataset_root}/test/labels"))

data.yaml: True
train/images: True
train/labels: True
val/images: True
val/labels: True
test/images: True
test/labels: True


In [6]:
from ultralytics import YOLO

model = YOLO("/kaggle/working/yolov8n_eca_neck_p3.yaml")
model.info()

YOLOv8n_eca_neck_p3 summary: 133 layers, 3,011,436 parameters, 3,011,420 gradients, 8.2 GFLOPs


(133, 3011436, 3011420, 8.1973248)

In [7]:
from ultralytics import YOLO

model = YOLO("/kaggle/working/yolov8n_eca_neck_p3.yaml")
model.load("yolov8n.pt")

results = model.train(
    data="/kaggle/working/data.yaml",
    epochs=80,
    imgsz=640,
    batch=8,
    device=0,
    workers=2,
    patience=25,
    project="/kaggle/working/runs",
    name="phi_yolov8n_eca_neck_p3",

    fliplr=0.3,
    flipud=0.15,
    mosaic=0.5,
    mixup=0.0,
    copy_paste=0.0,

    degrees=0.0,
    translate=0.02,
    scale=0.10,
    shear=0.0,
    perspective=0.0,

    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,

    plots=True,
    save=True,
    val=True
)

Transferred 210/356 items from pretrained weights
Ultralytics 8.4.33 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.3, flipud=0.15, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/working/yolov8n_eca_neck_p3.yaml, momentum=0.937, mosaic=0.5, multi_scale=0.0, name=phi_yolov8n_eca_neck_p3, nbs=64, nms=Fals